# 1장 2강: 유의수준·검정력·표본 크기의 트레이드오프와 실질적 유의성 — 실습문제

## 실습 목표

- 효과 크기와 유의수준이 같을 때 표본 크기에 따라 검정력이 어떻게 변하는지 설명할 수 있다.
- 목표 검정력과 유의수준을 이용해 필요한 표본 크기를 계산할 수 있다.
- 두 집단의 평균 차이에 대한 Cohen’s d를 계산하고 해석할 수 있다.
- p-value와 효과 크기, 실무 기준을 함께 사용하여 의사결정 근거를 작성할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing.csv`

주요 컬럼은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `CentralAir` | 중앙 냉방시설 유무(`Y`, `N`) |
| `KitchenQual` | 주방 품질(`Ex`, `Gd`, `TA`, `Fa`) |

> 모든 검정은 별도 지시가 없으면 유의수준 `α = 0.05`를 사용합니다.  
> Cohen’s d는 절댓값을 기준으로 약 0.2는 작은 효과, 0.5는 중간 효과, 0.8 이상은 큰 효과로 해석합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
"""
Filename: 01_lab_preparation.py
Description: Ames Housing 데이터셋 로드 및 무결성 검증 (.ipynb / .py 겸용)
"""

import math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.power import TTestIndPower

# 1. 파일 상대 경로 설정 (.py 와 .ipynb 모두 호환되는 표준 방식)
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    # 주피터 노트북(.ipynb) 환경에서는 현재 실행 중인 폴더(cwd)를 기준으로 설정
    ROOT = Path.cwd()

# 파일이 현재 폴더에 있거나, 하위 data/ 폴더에 위치하는지 순차 탐색
data_path = ROOT / "ames_housing.csv"
if not data_path.exists():
    data_path = ROOT / "data" / "ames_housing.csv"

# 2. 데이터 불러오기 (한글 인코딩 깨짐 방지 및 UTF-8 호환)
df = pd.read_csv(data_path, encoding="utf-8-sig")

# 3. 데이터 기본 정보 확인
print("=" * 60)
print("[데이터 기본 정보 확인]")
print(f"- 데이터 형상 (Rows, Columns): {df.shape}")
print(f"- 전체 결측치 수 (Total Missing Values): {df.isnull().sum().sum()} 건")
print(f"- 주요 컬럼 존재 여부:")
target_cols = ["SalePrice", "CentralAir", "KitchenQual"]
for col in target_cols:
    exists = col in df.columns
    print(f"  * {col}: {'존재함' if exists else '없음'}")
print("=" * 60)

# 상위 5개 행 주요 컬럼 출력
print("\n[상위 5개 행 미리보기]")
print(df[target_cols].head())

[데이터 기본 정보 확인]
- 데이터 형상 (Rows, Columns): (1460, 10)
- 전체 결측치 수 (Total Missing Values): 0 건
- 주요 컬럼 존재 여부:
  * SalePrice: 존재함
  * CentralAir: 존재함
  * KitchenQual: 존재함

[상위 5개 행 미리보기]
   SalePrice CentralAir KitchenQual
0     208500          Y          Gd
1     181500          Y          TA
2     223500          Y          Gd
3     140000          Y          Gd
4     250000          Y          Gd


---

## 필수 1. 표본 크기와 검정력 비교

### 문제 1-1. 중앙 냉방시설 비교 연구에 필요한 표본 수 설계

#### 문제 설명

중앙 냉방시설이 있는 주택과 없는 주택의 판매가격을 비교하는 연구를 준비하고 있습니다. 사전 조사에서 표준화 효과 크기는 `0.3` 정도로 예상했습니다.

다음 조건에서 그룹당 표본 크기에 따른 검정력을 비교하고, 목표 검정력 0.8을 확보하기 위해 필요한 표본 수를 계산하세요.

- 예상 효과 크기: `0.3`
- 유의수준: `0.05`
- 양측검정
- 비교할 그룹당 표본 수: `30명`, `100명`, `300명`

#### 요구사항

1. `TTestIndPower()` 객체를 생성하세요.
2. 그룹당 표본 수가 30명, 100명, 300명일 때의 검정력을 각각 계산하세요.
3. 표본 크기별 검정력을 소수점 셋째 자리까지 출력하세요.
4. 유의수준 0.05, 목표 검정력 0.8, 효과 크기 0.3일 때 그룹당 필요한 표본 수를 계산하세요.
5. 계산된 표본 수는 소수점 이하를 올림하여 정수로 출력하세요.
6. 세 표본 크기 중 목표 검정력 0.8을 충족하는 경우를 확인하세요.

#### 해석 질문

**Q1.** 효과 크기와 유의수준이 같을 때 표본 크기가 커지면 검정력은 어떻게 변하나요?  
**Q2.** 검정력이 낮으면 실제 차이가 존재할 때 어떤 오류의 위험이 커지나요?  
**Q3.** 필요한 표본 수를 소수점 이하 올림으로 처리하는 이유는 무엇인가요?  
**Q4.** 표본 수를 무조건 크게 설정하는 것이 항상 최선인가요?

#### 제출 결과

- 표본 크기별 검정력
- 목표 검정력에 필요한 그룹당 표본 수
- 목표 검정력 충족 여부
- Q1~Q4 답변

In [2]:
# 필수 1 코드를 작성하세요.

"""
Filename: 02_mandatory_1_power_analysis.py
Description: 표본 크기별 검정력 비교 및 목표 검정력 도달에 필요한 표본 크기 산출
"""

import math
from statsmodels.stats.power import TTestIndPower

# 1. 파워 분석 객체 생성
power_analysis = TTestIndPower()

# 2. 파라미터 정의
effect_size = 0.3  # 예상 표준화 효과 크기 (Cohen's d)
alpha = 0.05  # 유의수준
target_power = 0.80  # 목표 검정력
sample_sizes = [30, 100, 300]  # 그룹당 비교할 표본 크기

print("=" * 60)
print(
    f"[검정력 분석 결과] (효과 크기 d={effect_size}, 유의수준 α={alpha}, 양측검정)"
)
print("=" * 60)

# 3. 그룹당 표본 크기별 검정력 계산 및 출력
powers = {}
for n in sample_sizes:
    calc_power = power_analysis.power(
        effect_size=effect_size,
        nobs1=n,
        alpha=alpha,
        ratio=1.0,
        alternative="two-sided",
    )
    powers[n] = calc_power
    status = (
        "충족 (>= 0.80)" if calc_power >= target_power else "미달 (< 0.80)"
    )
    print(f"- 그룹당 {n:>3}명: 검정력 = {calc_power:.3f} [{status}]")

# 4. 목표 검정력 0.80을 달성하기 위한 그룹당 필요 표본 수 계산
n_required = power_analysis.solve_power(
    effect_size=effect_size,
    power=target_power,
    alpha=alpha,
    ratio=1.0,
    alternative="two-sided",
)
n_required_ceil = math.ceil(n_required)

print("-" * 60)
print(f"- 목표 검정력(0.80)을 달성하기 위해 필요한 그룹당 표본 수:")
print(f"  * 계산값: {n_required:.2f} 명")
print(f"  * 올림 적용 정수값: {n_required_ceil} 명 (그룹당 최소치)")
print(f"  * 양 집단 전체 필요 표본 수: {n_required_ceil * 2} 명")
print("=" * 60)

[검정력 분석 결과] (효과 크기 d=0.3, 유의수준 α=0.05, 양측검정)
- 그룹당  30명: 검정력 = 0.208 [미달 (< 0.80)]
- 그룹당 100명: 검정력 = 0.560 [미달 (< 0.80)]
- 그룹당 300명: 검정력 = 0.956 [충족 (>= 0.80)]
------------------------------------------------------------
- 목표 검정력(0.80)을 달성하기 위해 필요한 그룹당 표본 수:
  * 계산값: 175.38 명
  * 올림 적용 정수값: 176 명 (그룹당 최소치)
  * 양 집단 전체 필요 표본 수: 352 명


## 필수 1 답변 작성란

- **Q1. 효과 크기와 유의수준이 같을 때 표본 크기가 커지면 검정력은 어떻게 변하나요?**  
  * **답변:** 표본 크기가 커질수록 **검정력(Power)은 단조 증가(상승)**합니다. 표본 수가 늘어나면 표본평균의 불확실성인 표준오차($SE = \frac{s}{\sqrt{n}}$)가 감소하여 표본 분포의 폭이 좁아지므로, 귀무가설($H_0$) 하의 분포와 대립가설($H_1$) 하의 분포 간 겹치는 영역이 줄어들어 실제 존재하는 차이를 탐지할 확률이 비약적으로 높아집니다.
- **Q2. 검정력이 낮으면 실제 차이가 존재할 때 어떤 오류의 위험이 커지나요?**  
  * **답변:** **제2종 오류(Type II Error, $\beta$, 위음성/False Negative)**의 위험이 커집니다. 즉, 모집단에 실제로는 의미 있는 차이(또는 퀀트에서의 알파)가 분명히 존재함에도 불구하고, 표본 수가 부족하여 $p$-value가 유의수준(0.05)보다 크게 나와 "차이가 없다"고 잘못 결론 내리고 유효한 시그널을 놓쳐버리게 됩니다.
- **Q3. 필요한 표본 수를 소수점 이하 올림으로 처리하는 이유는 무엇인가요?**  
  * **답변:** 첫째, 표본(사람, 주택 수 등)은 물리적으로 쪼갤 수 없는 **이산적인 정수(Integer)**이기 때문입니다. 둘째, 계산된 값(예: 175.38명)을 버림(내림)하여 175명을 확보하면 실제 달성 검정력이 목표치인 0.80(80%)에 미달(예: 79.9%)하게 되므로, **보수적으로 목표 검정력(80%)을 엄밀히 보장하기 위해 반드시 올림(ceil)** 처리를 해야 합니다.
- **Q4. 표본 수를 무조건 크게 설정하는 것이 항상 최선인가요?**  
  * **답변:** **아닙니다.** 
    1. **비용 및 리소스 문제**: 데이터 수집, 정제, 설문 조사에는 막대한 시간과 비용이 듭니다.
    2. **통계적 유의성의 왜곡(p-hacking 함정)**: 표본 수가 수십만 건으로 과도하게 커지면, 비즈니스나 투자 관점에서는 $0.0001\%$처럼 아무런 실질적 의미가 없는 극미한 노이즈 차이조차 표준오차가 $0$에 수렴하면서 $p < 0.001$로 무차별 기각되는 **'사소한 차이의 유의성 왜곡'** 문제가 발생하기 때문입니다.

---

## 필수 2. 통계적 유의성과 실질적 유의성 종합 판단

### 문제 2-1. 중앙 냉방시설 유무에 따른 판매가격 차이 분석

#### 문제 설명

한 부동산 회사는 중앙 냉방시설이 있는 주택과 없는 주택의 평균 판매가격 차이를 분석하려고 합니다. 회사는 두 집단의 평균 판매가격 차이가 **100,000달러 이상**이어야 냉방시설 설치 지원 사업을 검토할 실무적 가치가 있다고 정했습니다.

#### 요구사항

1. `CentralAir == "Y"`인 주택의 `SalePrice`를 `air_yes`에 저장하세요.
2. `CentralAir == "N"`인 주택의 `SalePrice`를 `air_no`에 저장하세요.
3. 두 집단의 표본 수와 평균 판매가격을 출력하세요.
4. `stats.ttest_ind()`로 두 집단 평균 차이에 대한 양측검정을 수행하세요.
5. 제공된 공식에 따라 Cohen’s d 계산 함수를 작성하고 효과 크기를 구하세요.
6. 평균 차이, p-value, Cohen’s d를 출력하세요.
7. 다음 기준으로 결과를 판단하세요.
   - `p-value ≤ 0.05`: 통계적으로 유의함
   - `|Cohen’s d| ≥ 0.8`: 큰 효과
   - `|평균 차이| ≥ 100000`: 회사 기준에서 실질적으로 유의함
8. 세 결과를 종합하여 사업 검토 여부에 대한 근거를 작성하세요.

#### Cohen’s d 공식

두 집단의 평균 차이를 합동 표준편차로 나누어 계산합니다.

$$
d = \frac{\bar{x}_1 - \bar{x}_2}{s_p}
$$

$$
s_p = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}}
$$

- $\bar{x}_1$, $\bar{x}_2$: 각 집단의 평균
- $s_1$, $s_2$: 각 집단의 표본 표준편차
- $n_1$, $n_2$: 각 집단의 표본 수
- $s_p$: 합동 표준편차

#### 해석 질문

**Q1.** p-value와 Cohen’s d는 각각 어떤 정보를 제공하나요?  
**Q2.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
**Q3.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
**Q4.** 회사가 정한 100,000달러 기준에서 실질적으로 유의한가요?  
**Q5.** 통계적으로 유의하고 효과 크기가 크더라도 회사의 실무 기준을 충족하지 못할 수 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 독립표본 t검정 결과
- 평균 차이와 Cohen’s d
- 통계적·효과크기·회사 기준 판단
- 최종 의사결정 근거
- Q1~Q5 답변

In [3]:
# 필수 2 코드를 작성하세요.

"""
Filename: 03_mandatory_2_practical_significance.py
Description: 중앙 냉방시설(CentralAir) 유무에 따른 판매가격 독립표본 t-검정 및 Cohen's d 계산
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats


# 1. 파일 상대 경로 설정 (.py 와 .ipynb 모두 호환되는 표준 방식)
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    # 주피터 노트북(.ipynb) 환경에서는 현재 실행 중인 폴더(cwd)를 기준으로 설정
    ROOT = Path.cwd()
 # 파일이 현재 폴더에 있거나, 하위 data/ 폴더에 위치하는지 순차 탐색
data_path = ROOT / "ames_housing.csv"
if not data_path.exists():
    data_path = ROOT / "data" / "ames_housing.csv"

# 2. 데이터 불러오기 (한글 인코딩 깨짐 방지 및 UTF-8 호환)
df = pd.read_csv(data_path, encoding="utf-8-sig")

# 1 & 2. 데이터 분리
air_yes = df[df["CentralAir"] == "Y"]["SalePrice"].dropna()
air_no = df[df["CentralAir"] == "N"]["SalePrice"].dropna()

# 3. 기술 통계량 산출
n_yes, n_no = len(air_yes), len(air_no)
mean_yes, mean_no = air_yes.mean(), air_no.mean()
s_yes, s_no = air_yes.std(ddof=1), air_no.std(ddof=1)
mean_diff = mean_yes - mean_no


# 5. Cohen's d 계산 함수 정의
def calculate_cohens_d(x1: pd.Series, x2: pd.Series) -> tuple[float, float]:
    n1, n2 = len(x1), len(x2)
    s1, s2 = x1.std(ddof=1), x2.std(ddof=1)
    # 합동 표준편차 (Pooled Standard Deviation)
    sp = np.sqrt(((n1 - 1) * (s1**2) + (n2 - 1) * (s2**2)) / (n1 + n2 - 2))
    # Cohen's d
    d = (x1.mean() - x2.mean()) / sp
    return d, sp


d_value, sp_value = calculate_cohens_d(air_yes, air_no)

# 4. 독립표본 t-검정 (등분산 가정 검정: Welch's t-test 권장 및 기본 Student t-test 병행)
# 문제 요구사항에 따라 전통적 합동 t-test 수행 (equal_var=True)
t_stat, p_val = stats.ttest_ind(air_yes, air_no, equal_var=True)

# 7. 기준 판정
stat_sig = p_val <= 0.05
large_effect = abs(d_value) >= 0.8
practical_sig = abs(mean_diff) >= 100000

print("=" * 65)
print("[중앙 냉방시설(CentralAir) 유무에 따른 주택가격 비교 분석]")
print("=" * 65)
print(f"1. 집단별 표본 통계:")
print(f"  - CentralAir 'Y': n = {n_yes:>5} 건, 평균 = ${mean_yes:>10,.2f}")
print(f"  - CentralAir 'N': n = {n_no:>5} 건, 평균 = ${mean_no:>10,.2f}")
print(f"  - 평균 차이 (Y - N): ${mean_diff:>10,.2f}")
print(f"  - 합동 표준편차(Sp) : ${sp_value:>10,.2f}")
print("-" * 65)
print(f"2. 통계적 가설검정 (독립표본 t-검정):")
print(f"  - t-통계량 (t-statistic) : {t_stat:.4f}")
print(f"  - p-값 (p-value)         : {p_val:.4e}")
print(
    f"  - 통계적 유의성 판정     : {'[유의함] (p <= 0.05)' if stat_sig else '[기각 실패]'}"
)
print("-" * 65)
print(f"3. 효과 크기 및 실무 기준 판정:")
print(f"  - Cohen's d 효과 크기    : {d_value:.4f}")
print(
    f"  - 효과 크기 판정         : {'[큰 효과] (|d| >= 0.8)' if large_effect else '[중간 이하 효과]'}"
)
print(
    f"  - 회사 실무 기준 ($10만) : {'[충족] 실질적 유의함' if practical_sig else '[미달] 실질적 유의성 부족'}"
)
print("=" * 65)

[중앙 냉방시설(CentralAir) 유무에 따른 주택가격 비교 분석]
1. 집단별 표본 통계:
  - CentralAir 'Y': n =  1365 건, 평균 = $186,186.71
  - CentralAir 'N': n =    95 건, 평균 = $105,264.07
  - 평균 차이 (Y - N): $ 80,922.64
  - 합동 표준편차(Sp) : $ 76,918.92
-----------------------------------------------------------------
2. 통계적 가설검정 (독립표본 t-검정):
  - t-통계량 (t-statistic) : 9.9149
  - p-값 (p-value)         : 1.8095e-22
  - 통계적 유의성 판정     : [유의함] (p <= 0.05)
-----------------------------------------------------------------
3. 효과 크기 및 실무 기준 판정:
  - Cohen's d 효과 크기    : 1.0521
  - 효과 크기 판정         : [큰 효과] (|d| >= 0.8)
  - 회사 실무 기준 ($10만) : [미달] 실질적 유의성 부족


### 필수 2 답변 작성란

- **Q1. p-value와 Cohen’s d는 각각 어떤 정보를 제공하나요?**  
  * **답변:** 
    * **p-value**: "두 집단 간에 실제 차이가 없다는 귀무가설($H_0$) 하에서, 현재 관측된 만큼 극단적인 평균 차이가 **순전히 표본 추출 상의 우연(Noise)으로 발생할 확률**"을 나타냅니다. (효과의 존재 유무/확률적 유의성)
    * **Cohen’s d**: 표본 수($N$)의 크기와 무관하게, 집단 간 평균 차이를 표준편차 단위로 정규화한 **"순수한 신호(Signal)의 절대적 크기(표준화된 효과 크기)"**를 제공합니다.
- **Q2. 두 집단의 판매가격 차이는 통계적으로 유의한가요?**  
  * **답변:** **네, 극단적으로 통계적으로 유의합니다.** $p$-value가 $10^{-15}$ 미만(거의 $0$)으로 유의수준 $\alpha = 0.05$보다 훨씬 작으므로 귀무가설을 강력하게 기각합니다.
- **Q3. Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?**  
  * **답변:** **큰 효과(Large Effect)**입니다. 계산된 Cohen's d 값은 약 $1.00 \sim 1.10$ 수준으로, 기준치인 $0.80$을 훌쩍 넘어서며 냉방시설 유무에 따른 주택가격 분포의 격차가 표준편차 1개 이상 벌어져 있음을 나타냅니다.
- **Q4. 회사가 정한 100,000달러 기준에서 실질적으로 유의한가요?**  
  * **답변:** **아닙니다 (미달).** 실제 관측된 두 집단 간의 평균 가격 차이는 약 **$75,000 ~ $85,000 달러** 수준으로, 회사가 설치 지원 사업의 경제적 손익분기점으로 설정한 **최소 허용 기준 $100,000달러에 도달하지 못합니다.**
- **Q5. 통계적으로 유의하고 효과 크기가 크더라도 회사의 실무 기준을 충족하지 못할 수 있나요?**  
  * **답변:** **네, 충분히 발생할 수 있으며 이것이 본 실습의 핵심 교훈입니다.** 통계적 유의성($p$-value)과 표준화된 효과 크기(Cohen's d)는 '데이터 자체의 분포 특성'을 평가하는 지표일 뿐, 비즈니스의 원가, 자본 조달 비용, 시공 수수료 등의 **현실적인 경제적 효용(Economic/Practical Criterion)**을 대변하지 못합니다. 따라서 $p < 0.05$이고 $d \ge 0.8$이더라도 회사의 절대적 금액 기준을 만족하지 못하면 사업 추진이 기각될 수 있습니다.

---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질이 `Gd`인 집단과 `TA`인 집단 비교

#### 문제 설명

부동산 중개회사는 주방 품질이 `Gd`(Good)인 주택과 `TA`(Typical/Average)인 주택의 평균 판매가격을 비교하려고 합니다. 회사는 평균 판매가격 차이가 **50,000달러 이상**이면 마케팅에서 주방 품질의 차이를 강조할 실무적 가치가 있다고 판단합니다.

> 필수 2에서 학습한 검정과 효과 크기 계산 절차를 새로운 두 집단에 적용하는 과제입니다.

#### 요구사항

1. 주방 품질이 `Gd`인 집단과 `TA`인 집단의 `SalePrice`를 각각 준비하세요.
2. 두 집단의 표본 수와 평균 판매가격을 출력하세요.
3. 두 집단의 평균 차이에 대한 양측 독립표본 t검정을 수행하세요.
4. Cohen’s d를 계산하세요.
5. 평균 차이, p-value, Cohen’s d를 출력하세요.
6. 다음 기준에 따라 각각 판단하세요.
   - 통계적 유의성: `p-value ≤ 0.05`
   - 큰 효과: `|Cohen’s d| ≥ 0.8`
   - 실무적 유의성: `|평균 차이| ≥ 50000`
7. 세 가지 판단을 종합하여 주방 품질을 마케팅에서 강조할 근거가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
**Q2.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
**Q3.** 회사가 정한 50,000달러 기준을 충족하나요?  
**Q4.** 최종적으로 주방 품질 차이를 마케팅에서 강조할 근거가 있다고 볼 수 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- t통계량과 p-value
- 평균 차이와 Cohen’s d
- 통계적·실질적 유의성 판단
- 최종 결론
- Q1~Q4 답변

In [4]:
# 과제 코드를 작성하세요.

"""
Filename: 04_assignment_kitchen_quality.py
Description: 주방 품질(KitchenQual 'Gd' vs 'TA')에 따른 주택 판매가격 t-검정 및 실무성 평가
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# 1. 파일 상대 경로 설정 (.py 와 .ipynb 모두 호환되는 표준 방식)
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    # 주피터 노트북(.ipynb) 환경에서는 현재 실행 중인 폴더(cwd)를 기준으로 설정
    ROOT = Path.cwd()

# 파일이 현재 폴더에 있거나, 하위 data/ 폴더에 위치하는지 순차 탐색
data_path = ROOT / "ames_housing.csv"
if not data_path.exists():
    data_path = ROOT / "data" / "ames_housing.csv"

# 2. 데이터 불러오기 (한글 인코딩 깨짐 방지 및 UTF-8 호환)
df = pd.read_csv(data_path, encoding="utf-8-sig")

# 1. 집단 데이터 준비
price_gd = df[df["KitchenQual"] == "Gd"]["SalePrice"].dropna()
price_ta = df[df["KitchenQual"] == "TA"]["SalePrice"].dropna()

# 2. 표본 통계치 산출
n_gd, n_ta = len(price_gd), len(price_ta)
mean_gd, mean_ta = price_gd.mean(), price_ta.mean()
s_gd, s_ta = price_gd.std(ddof=1), price_ta.std(ddof=1)
mean_diff = mean_gd - mean_ta

# 3. 독립표본 t-검정
t_stat, p_val = stats.ttest_ind(price_gd, price_ta, equal_var=True)

# 4. Cohen's d 계산
sp = np.sqrt(((n_gd - 1) * (s_gd**2) + (n_ta - 1) * (s_ta**2)) / (n_gd + n_ta - 2))
d_value = mean_diff / sp

# 5 & 6. 기준 판단
stat_sig = p_val <= 0.05
large_effect = abs(d_value) >= 0.8
practical_sig = abs(mean_diff) >= 50000

print("=" * 65)
print("[주방 품질(KitchenQual: Good vs Typical)에 따른 판매가격 비교]")
print("=" * 65)
print(f"1. 집단별 표본 통계:")
print(f"  - KitchenQual 'Gd' (Good)    : n = {n_gd:>5} 건, 평균 = ${mean_gd:>10,.2f}")
print(f"  - KitchenQual 'TA' (Typical) : n = {n_ta:>5} 건, 평균 = ${mean_ta:>10,.2f}")
print(f"  - 평균 판매가격 차이 (Gd - TA): ${mean_diff:>10,.2f}")
print(f"  - 합동 표준편차 (Sp)         : ${sp:>10,.2f}")
print("-" * 65)
print(f"2. 통계적 유의성 검정:")
print(f"  - t-통계량 (t-statistic)     : {t_stat:.4f}")
print(f"  - p-값 (p-value)             : {p_val:.4e}")
print(
    f"  - 통계적 유의성 판정         : {'[유의함] (p <= 0.05)' if stat_sig else '[유의하지 않음]'}"
)
print("-" * 65)
print(f"3. 효과 크기 및 실무 기준 판정:")
print(f"  - Cohen's d 효과 크기        : {d_value:.4f}")
print(
    f"  - 효과 크기 판정             : {'[큰 효과] (|d| >= 0.8)' if large_effect else '[보통/작은 효과]'}"
)
print(
    f"  - 실무 기준 ($50,000 이상)   : {'[충족] 실질적 가치 충분' if practical_sig else '[미달] 가치 부족'}"
)
print("=" * 65)

[주방 품질(KitchenQual: Good vs Typical)에 따른 판매가격 비교]
1. 집단별 표본 통계:
  - KitchenQual 'Gd' (Good)    : n =   586 건, 평균 = $212,116.02
  - KitchenQual 'TA' (Typical) : n =   735 건, 평균 = $139,962.51
  - 평균 판매가격 차이 (Gd - TA): $ 72,153.51
  - 합동 표준편차 (Sp)         : $ 51,572.36
-----------------------------------------------------------------
2. 통계적 유의성 검정:
  - t-통계량 (t-statistic)     : 25.2628
  - p-값 (p-value)             : 3.5568e-115
  - 통계적 유의성 판정         : [유의함] (p <= 0.05)
-----------------------------------------------------------------
3. 효과 크기 및 실무 기준 판정:
  - Cohen's d 효과 크기        : 1.3991
  - 효과 크기 판정             : [큰 효과] (|d| >= 0.8)
  - 실무 기준 ($50,000 이상)   : [충족] 실질적 가치 충분


### 과제 답변 작성란

- **Q1. 두 집단의 판매가격 차이는 통계적으로 유의한가요?**  
  * **답변:** **네, 매우 통계적으로 유의합니다.** 실제 검정 결과 $p$-value $\approx 3.56 \times 10^{-115}$로 $0.05$보다 극단적으로 작으므로, Gd 등급과 TA 등급 간의 가격 차이가 우연에 의해 발생했을 확률은 사실상 $0$에 가깝습니다.
- **Q2. Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?**  
  * **답변:** **큰 효과(Large Effect)**입니다. 실제 계산된 Cohen's d 값은 **약 $1.40$ (1.3991)** 수준으로, 기준치인 $0.80$을 크게 상회합니다. 주방 품질 등급 간의 주택가격 차이가 집단 내 변동성에 비해 매우 뚜렷하게 분리되어 있음을 의미합니다.
- **Q3. 회사가 정한 50,000달러 기준을 충족하나요?**  
  * **답변:** **네, 여유 있게 초과 충족합니다.** 실제 관측된 두 집단 간 평균 가격 차이는 **약 $72,153.51달러**(Gd 평균 $212,116.02 - TA 평균 $139,962.51)로, 회사의 마케팅 실무 기준선인 $50,000달러를 $22,000달러 이상 상회합니다.
- **Q4. 최종적으로 주방 품질 차이를 마케팅에서 강조할 근거가 있다고 볼 수 있나요?**  
  * **답변:** **네, 마케팅 소구점으로 적극 활용할 타당한 근거가 충분합니다.** 
    1. **통계적 확실성($p < 0.05$)**: 표본 오차에 의한 착시가 아님이 증명됨.
    2. **표준화된 격차($d > 0.8$)**: 전반적인 주택 시장 분포상에서도 확고한 우위를 점함.
    3. **실무적 경제성($\Delta \ge \$50,000$)**: 실거래가에서 평균 7만 달러 이상의 프리미엄이 확인되므로, 리모델링 투자 대비 가치 상승을 강조하는 세일즈 마케팅 전략을 집행하기에 매우 타당합니다.

## 5. 실습 마무리 답변

1. **검정력은 무엇이며 제2종 오류와 어떤 관계가 있나요?**
   * **답변:** 검정력(Statistical Power, $1 - \beta$)은 **"실제 대립가설($H_1$)이 참일 때(즉, 효과나 차이가 실제로 존재할 때), 귀무가설을 올바르게 기각하고 그 차이를 탐지해낼 확률"**입니다. 제2종 오류($\beta$)는 실제 차이가 존재함에도 기각하지 못하는 오류(False Negative)이므로, 검정력과 제2종 오류는 합이 1인 **상호 배타적 역의 관계($\text{Power} = 1 - \beta$)**를 갖습니다.

2. **같은 효과 크기와 유의수준에서 표본 크기가 커지면 검정력은 어떻게 변하나요?**
   * **답변:** **검정력은 단조 증가합니다.** 표본 크기($N$)가 커지면 표본평균의 표준오차($SE = s/\sqrt{N}$)가 줄어들어 검정통계량의 분포가 좁고 뾰족해지며, 이에 따라 귀무가설의 기각역에 도달할 확률이 비약적으로 높아지기 때문입니다.

3. **목표 검정력이 높거나 발견하려는 효과가 작을수록 필요한 표본 수는 어떻게 변하나요?**
   * **답변:** **필요한 표본 수는 기하급수적으로 증가합니다.** 
     * 미세한 차이(작은 효과 크기, 작은 신호)를 노이즈와 구별해내려면 표준오차를 극도로 낮춰야 하며, 
     * 탐지 성공률(목표 검정력 $1-\beta$)을 80%에서 90%, 95%로 올리기 위해서는 대립가설 분포의 기각역 면적을 넓혀야 하므로 표본 크기 $N$이 대폭 커져야 합니다. ($N \propto 1 / d^2$)

4. **p-value와 Cohen’s d를 함께 확인해야 하는 이유는 무엇인가요?**
   * **답변:** **$p$-value는 '표본 수($N$)의 함수'**이기 때문에, 표본이 수만~수십만 건으로 매우 커지면 실질적으로 아무 의미 없는 $0.001$의 미세한 차이도 $p < 0.001$로 유의하게 도출됩니다. 반면 **Cohen’s d는 표본 크기($N$)의 영향을 배제하고 순수한 '신호 대 잡음비(효과의 상대적 크기)'만을 측정**합니다. 따라서 $p$-value로는 "이 결과가 우연이 아닌가?"를 검증하고, Cohen’s d로는 "이 차이가 실제로 의미 있게 큰가?"를 상호 보완적으로 교차 검증해야 합니다.

5. **통계적으로 유의한 결과가 반드시 실무적으로 중요한 결과를 의미하나요?**
   * **답변:** **절대 그렇지 않습니다 (통계적 유의성 $\neq$ 경제적/실무적 유의성).** 
     * 필수 2번 실습처럼 통계적으로 유의($p < 0.001$)하고 표준화 효과가 크더라도($d > 0.8$), 회사가 감당해야 할 원가나 사업 손익분기점(예: 10만 달러)을 넘지 못하면 비즈니스 관점에서는 채택할 수 없습니다. 
     * 데이터 사이언티스트와 퀀트는 $p < 0.05$라는 통계적 지표에만 매몰되지 않고, 항상 비즈니스 도메인의 실질적 임계값(손익분기점, 슬리피지, 거래 비용 등)을 반드시 결합하여 최종 의사결정을 내려야 합니다.